## imports

In [ ]:
from astroquery.mast import Observations
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval

In [2]:
from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs

In [3]:
import numpy as np

## An Aside Running a MAST Query

we want to search MAST.

here are a few common parameters you may want to use when searching MAST:

| Criterion | Meaning |
|--|--|
|`object_name`| The name of the celestial object: this name will be resolved to coordinates|
|`target_name`| The name of the celestial object **as entered by the proposer**|
|`obs_collection`| Roughly equivalent to mission |

Searching by `object_name` will tend to be a slow filter, since the API must:
1. Query a resolver service (likely Simbad or NED) to transform your input into coordinates
2. Look for observations with a spatial match on your input. This is no small task when the database has over 500 million rows!

TO-DO: talk about setting the resolver manually


Let's run an example query:

In [ ]:
obs = Observations.query_criteria(
    object_name="Trappist-1",
    obs_collection="JWST"
    )

len(obs)

Lots of observations. Let's take a look at what the proposers called the targets:

In [ ]:
set(obs['target_name'])

These results highlight the pitfalls of searching by target name. Sometimes an observer may use a non-standard name, or a the name might not be populated in the database at all!

You can sometimes get around this with wildcard characters:

TO-DO: repeat the above but with `target_name` and a wildcard character. {show an example of a wildcard match before, just in text}

In [ ]:
obs = Observations.query_criteria(
    target_name="Trappist*",
    obs_collection="JWST"
    )

len(obs)

TO-DO: very fast but we're missing 17 observations

## Narrowing in on Our Brick

If we're careful, we can take advantage of the `target_name` trick for very fast searching.

In [ ]:
obs = \
Observations.query_criteria(
    target_name="M31-B01-F*-UVIS",
    wave_region="UV",
    proposal_id=12058,
    project="HST"
                           )

In [ ]:
obs

In [ ]:
prod = Observations.get_product_list(obs)

In [ ]:
prod

SHOW A PREVIEW IMAGE

In [ ]:
filt = Observations.filter_products(prod, mrp_only=True, productSubGroupDescription="DRZ")

In [ ]:
filt

In [ ]:
curis = Observations.get_cloud_uris(filt)

curis

In [ ]:
curis = Observations.get_cloud_uris(filt)

# need "anon":true to anonymously access the cloud files
hdus = [fits.open(f, fsspec_kwargs={"anon": True}) for f in curis]

In [ ]:
hdus

In [ ]:
# Create a figure on which to plot our data
fig = plt.figure(0, [9, 9])
ax = fig.add_subplot(111)

# Get the primary data from the first fits file
test_data = hdus[0][1].data

# Automatically scale the brightness based on the data
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(test_data)

# Show our data with the scaling from above
ax.imshow(test_data, vmin=lims[0], vmax=lims[1], cmap='BuPu_r')

In [10]:
wcs, shape = find_optimal_celestial_wcs(hdus, hdu_in='SCI')

In [ ]:
# Create an empty "maps" list to hold our outputs
maps = []

# Loop through the hdus, getting the data and header from each
for hdu in hdus:
    fmap = hdu[1].data
    fhead = hdu[1].header

    regmap, foot = reproject_interp((fmap, fhead), wcs, shape)

    # append the reprojected map to a list
    maps.append(regmap)
        
# average all of the reprojected maps together
combined = np.nanmean(np.dstack((maps)), 2)

In [ ]:
from reproject.mosaicking import reproject_and_coadd

In [ ]:
sci_hdus = [hdu[1].data for hdu in hdus]

KeyboardInterrupt: 

In [ ]:
wcs, shape = find_optimal_celestial_wcs(hdu_tuples, resolution=1*u.arcsec)

In [ ]:
array, footprint = reproject_and_coadd(hdu_tuples,
                                       wcs, shape,
                                       reproject_function=reproject_interp,
                                       match_background=True
                                      )

In [ ]:
reproject_and_coadd?

In [20]:
array, footprint = reproject_and_coadd(hdus,
                                       wcs, shape,
                                       hdu_in="SCI",
                                       reproject_function=reproject_interp
                                      )

FileNotFoundError: [Errno 2] No such file or directory: "<class 's3fs.core.S3File'>"

In [15]:
hdus

[[<astropy.io.fits.hdu.image.PrimaryHDU object at 0x113f55be0>, <astropy.io.fits.hdu.image.ImageHDU object at 0x113f556a0>, <astropy.io.fits.hdu.image.ImageHDU object at 0x11406ee90>, <astropy.io.fits.hdu.image.ImageHDU object at 0x11406f110>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x113f55a90>],
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDUList'> (partially read),
 <class 'astropy.io.fits.hdu.hdulist.HDULi

In [ ]:
# Create the figure for the new map
fig = plt.figure(0, [9, 9])
ax = fig.add_subplot(111, projection=wcs)

# Automatically scale the brightness
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(combined)

# Plot the new map
ax.imshow(combined, vmin=lims[0], vmax=lims[1], origin='lower', cmap='BuPu_r')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')

In [ ]:
# Create the figure for the new map
fig = plt.figure(0, [9, 9])
ax = fig.add_subplot(111, projection=wcs)

# Automatically scale the brightness
# interval = ZScaleInterval(contrast=0.4)
# lims = interval.get_limits(combined)

# Plot the new map
# ax.imshow(combined, vmin=lims[0], vmax=lims[1], origin='lower', cmap='BuPu_r')
ax.imshow(combined)
ax.set_xlabel('RA')
ax.set_ylabel('Dec')

In [ ]:
combined